In [1]:
import pandas as pd
df = pd.read_csv('top_1000_task_messages.csv')
df.head()

,conversation_id,timestamp,user,text,source_file,cleaned_text,task_score,heuristic_score,combined_score
0,258,2017-07-12T12:33:06.416727,Kareen,<@Sidney> what's the issue specifically btw? s...,merged-clojurians-2017.xml,<@Sidney> what's the issue specifically btw? s...,0.224235,5,0.406964
1,5267,2018-10-21T15:18:02.000100,Cyndi,Can someone please review this PullRequest of ...,merged-elmlang-beginners-2018.xml,Can someone please review this PullRequest of ...,0.277982,4,0.394588
2,790,2018-09-12T14:22:54.000100,Monserrate,<@Melodee> can you post the code? that's almos...,merged-racket-general-2018.xml,<@Melodee> can you post the code? that's almos...,0.270232,4,0.389162
3,8,2017-06-21T09:35:22.245969,Candi,<@Kenia> can you file an issue or PR on that r...,merged-clojurians-2017.xml,<@Kenia> can you file an issue or PR on that r...,0.250143,4,0.375100
4,6049,2018-11-25T14:33:26.089000,Pura,I saw you created an issue there. Can you upda...,merged-elmlang-beginners-2018.xml,I saw you created an issue there. Can you upda...,0.246988,4,0.372891


In [2]:
# Load environment variables
from dotenv import load_dotenv
import os
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [3]:
import os
import json
import re
from datetime import datetime
import google.generativeai as genai

JSON_KEYS = ("task_summary", "assignee", "issue_creation_date")
_json_decoder = json.JSONDecoder()

def _gemini_generate_json(prompt: str,
                          model_name: str = "models/gemini-2.0-flash-live-001",
                          response_mime_type: str = "application/json") -> str:
    """
    Calls Gemini and requests JSON-only output. Returns the raw text response.
    You can switch to 'gemini-1.5-pro' for heavier tasks.
    """
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY not set")

    genai.configure(api_key=api_key)
    model = genai.GenerativeModel(
        model_name,
        generation_config={
            "temperature": 0.2,
            "top_p": 0.9,
            "top_k": 40,
            "max_output_tokens": 512,
            "response_mime_type": response_mime_type,  # <- ask for JSON
        },
        # (optional) you can add a system instruction if you like:
        system_instruction="You return only strict JSON. No code fences. No extra prose."
    )
    resp = model.generate_content(prompt)
    # Gemini responses put text in resp.text
    return resp.text or ""

In [ ]:
import os
import json
import re
from datetime import datetime
import google.generativeai as genai

JSON_KEYS = ("task_summary", "assignee", "issue_creation_date")
_json_decoder = json.JSONDecoder()

def _parse_first_json(s: str):
    """
    Extract the first top-level JSON object from an arbitrary string.
    Returns a Python object or raises ValueError if none found.
    """
    if not s:
        raise ValueError("empty string")

    i = s.find("{")
    if i == -1:
        raise ValueError("no JSON object start")
    while i != -1:
        try:
            obj, end = _json_decoder.raw_decode(s[i:])
            return obj
        except json.JSONDecodeError:
            i = s.find("{", i + 1)
    raise ValueError("could not parse JSON")

def _normalize_fields(obj: dict, row) -> dict:
    out = {k: obj.get(k, None) for k in JSON_KEYS}
    out["issue_creation_date"] = row["timestamp"]

    for k, v in out.items():
        if isinstance(v, str) and v.strip() == "":
            out[k] = None

    if out["assignee"] is not None:
        a = out["assignee"].strip()
        a = re.sub(r"^<@([^>]+)>$", r"\1", a)
        a = re.sub(r"^@+", "", a)
        out["assignee"] = a if a else None

    if out["task_summary"] is not None and len(out["task_summary"]) > 120:
        out["task_summary"] = out["task_summary"][:120].rstrip()
    return out

def _gemini_generate_json(prompt: str,
                          model_name: str = "models/gemini-2.0-flash-live-001",
                          response_mime_type: str = "application/json") -> str:
    """
    Calls Gemini and requests JSON-only output. Returns the raw text response.
    You can switch to 'gemini-1.5-pro' for heavier tasks.
    """
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY not set")

    genai.configure(api_key=api_key)

    model = genai.GenerativeModel(
        model_name,
        generation_config={
            "temperature": 0.2,
            "top_p": 0.9,
            "top_k": 40,
            "max_output_tokens": 512,
            "response_mime_type": response_mime_type,  # <- ask for JSON
        },
        system_instruction="You return only strict JSON. No code fences. No extra prose. If uncertain, keep fields minimal and truthful (use null, not N/A)."
    )
    resp = model.generate_content(prompt)
    # Gemini responses put text in resp.text
    return resp.text or ""

def extract_task_info(row):
    """
    Extracts task fields for Jira from a chat row using Gemini.
    Fields: task_summary, assignee, issue_creation_date.
    """
    text = row["cleaned_text"]
    date = row["timestamp"]  # ISO 8601
    convo_id = row.get("conversation_id")
    author = row.get("user")
    source = row.get("source_file")

    prompt = f"""
You are an expert Jira triage assistant. Return ONLY strict JSON; no prose, no backticks.
INPUT
- message_text: {text}
- message_timestamp: {date}

EXTRACTION RULES
1) task_summary
   - Summarize the specific actionable request or problem in <= 120 characters from the message_text.
   - Prefer imperative phrasing when appropriate.
   - If the message is only a question, summarize the implied action.
   - If multiple actions exist, choose the most central one.
   - Keep it self-contained (no @mentions, no links).

2) assignee
   - Extract a single user if explicitly mentioned after @ (Slack formats like <@Sidney>, @sidney, @sidney-b, or @First Last) in message_text.
   - If no explicit assignee, return null.

3) issue_creation_date
   - Always return the provided message_timestamp as-is.

4) General
   - Treat explicit review/approval, bugfix, code/logs, or answering a blocker as a task.
   - Do NOT add fields not requested.
   - If uncertain, keep fields minimal and truthful (use null, not "N/A").

OUTPUT FORMAT
Return ONLY strict JSON (no code fences, no extra text).
Keys: "task_summary", "assignee", "issue_creation_date".
"""

    try:
        raw = _gemini_generate_json(prompt, model_name='models/gemini-2.5-flash-lite')  # or "gemini-1.5-pro"
        parsed = _parse_first_json(raw)
    except Exception as e:
        # If Gemini output is unusable, fall back to minimal safe skeleton.
        print(e)
        parsed = {"task_summary": None, "assignee": None, "issue_creation_date": date}

    safe = _normalize_fields(parsed, row)
    # Ensure all keys exist & are JSON-serializable
    safe = {k: safe.get(k, None) for k in JSON_KEYS}
    return safe


In [10]:
os.makedirs(os.path.join(os.getcwd(), "labels"), exist_ok=True)

In [ ]:
# ...existing code...
import time
import json
import os

BATCH_SIZE = 5
SECONDS_PER_BATCH = 60
MAX_RETRIES = 3
RETRY_BACKOFF_BASE = 2  # seconds

rows = df.to_dict(orient="records")
start = 500
end = 1000
# total = len(rows)
extracted_list = []

def _safe_extract(row):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return extract_task_info(row)
        except Exception as e:
            print(f"extract_task_info failed (attempt {attempt}/{MAX_RETRIES}) for timestamp={row.get('timestamp')}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_BASE ** (attempt - 1))
    # final fallback
    return {"task_summary": None, "assignee": None, "issue_creation_date": row.get("timestamp")}

count = 1
for start in range(start, end, BATCH_SIZE):
    batch = rows[start:start + BATCH_SIZE]
    batch_results = []
    for row in batch:
        batch_results.append(_safe_extract(row))
    extracted_list.extend(batch_results)

    # atomic-ish write: write to temp then replace
    tmp_path = os.path.join("labels", f"jira_tasks_{count}.json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(batch_results, f, indent=2)
    os.replace(tmp_path, os.path.join("labels", f"jira_tasks_{(processed//BATCH_SIZE)}.json"))

    processed = min(start + BATCH_SIZE, end)
    print(f"Processed {processed}/{end} rows.")
    if processed < end:
        print(f"Sleeping {SECONDS_PER_BATCH} seconds to respect rate limits...")
        time.sleep(SECONDS_PER_BATCH)
    count += 1

# final ensure file present
with open(os.path.join("labels", f"jira_tasks_first_{end}.json"), "w") as f:
    json.dump(extracted_list, f, indent=2)

print("All done.")


Processed 500/500 rows.
All done.


In [12]:
import json
tasks = []
for i in range(1, 101):
    fname = f'labels/jira_tasks_{i}.json'
    if not os.path.exists(fname):
        continue
    with open(fname, 'r') as f:
        data = json.load(f)
        tasks.extend(data)
print(f'Total tasks extracted: {len(tasks)}')
with open('labels/jira_tasks_250_500.json', 'w') as f:
    json.dump(tasks, f, indent=2)

Total tasks extracted: 255


In [13]:
all_tasks = []
with open('labels/jira_tasks_first_250.json', 'r') as f:
    data = json.load(f)
    all_tasks.extend(data)
with open('labels/jira_tasks_250_500.json', 'r') as f:
    data = json.load(f)
    all_tasks.extend(data)
with open('labels/jira_tasks_first_500.json', 'w') as f:
    json.dump(all_tasks, f, indent=2)